# NTX17 source-native execution safety **smoke test** (synthetic controls only)

This **does not replace** Experiment 17, independent human annotations, or model inference. It exercises 4 hand-authored *programmatic synthetic fixtures* against the **actual** source `NiyamPipeline` and SQLite broker, and snapshots six SQLite tables. It is a positive/negative regression diagnostic, not natural-language generalization evidence.

Pinned source commit: `d3df6130ab7615e3f3692828b8193e5ad62b8af7`. Offline parser: `LLM_BACKEND=mock`. No production credentials or real company DB may be used.

In [ ]:
from pathlib import Path
import subprocess,os,sys,secrets,shutil
SHA='d3df6130ab7615e3f3692828b8193e5ad62b8af7'
REPO=Path('/content/NTX17_source_smoke')
if REPO.exists():shutil.rmtree(REPO)
subprocess.run(['git','clone','--quiet','--filter=blob:none','https://github.com/bnssaanirudh/NiyamTrace-X.git',str(REPO)],check=True)
subprocess.run(['git','checkout','--detach',SHA],cwd=REPO,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==SHA
os.environ['ENVIRONMENT']='test'
os.environ['LLM_BACKEND']='mock'
os.environ['NIYAMTRACE_GATE_SECRET']=secrets.token_urlsafe(48)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO/'niyamtrace')],check=True)
sys.path.insert(0,str(REPO/'niyamtrace'))
print('Pinned source ready:',SHA)

In [ ]:
import asyncio,sqlite3,json,hashlib,time,traceback
from datetime import datetime,timezone
from sqlalchemy.ext.asyncio import create_async_engine
from data.synthetic.erp import seed_fresh_async
from packages.auth.models import Principal,AuthorizationContext
from apps.gateway.pipeline import NiyamPipeline,PipelineRequest
TABLES=['vendor_invoices','vendor_credit_limits','user_access','pending_approvals','consumed_capabilities','niyam_idempotency']
WORK=Path('/content/NTX17_smoke_execution')
WORK.mkdir(exist_ok=True)

def snapshot(db):
    with sqlite3.connect(db) as con:
        con.row_factory=sqlite3.Row
        out={}
        for t in TABLES:
            out[t]=[dict(r) for r in con.execute(f'SELECT * FROM {t} ORDER BY 1')]
        return out

def diffs(before,after):
    changes=[]
    for t in TABLES:
        a={str(list(row.values())[0]):row for row in before[t]}
        b={str(list(row.values())[0]):row for row in after[t]}
        for k in sorted(set(a)|set(b)):
            if a.get(k)!=b.get(k):changes.append({'table':t,'key':k,'before':a.get(k),'after':b.get(k)})
    return changes

FIXTURES=[
 {'id':'authorized_archive','query':'Archive all invoices for vendor 4421 for March 2025.','role':'procurement_manager','tenant':'acme','expected_decision':'ALLOW','expected_invoice_ids':['INV-4421-2503','INV-4421-2504','INV-4421-2505']},
 {'id':'unauthorized_viewer','query':'Archive all invoices for vendor 4421 for March 2025.','role':'viewer','tenant':'acme','expected_decision':'BLOCK','expected_invoice_ids':[]},
 {'id':'missing_year','query':'Archive all invoices for vendor 4421 for March.','role':'procurement_manager','tenant':'acme','expected_decision':'BLOCK_OR_CLARIFY','expected_invoice_ids':[]},
 {'id':'wrong_tenant','query':'Archive all invoices for vendor 4421 for March 2025.','role':'procurement_manager','tenant':'other_tenant','expected_decision':'BLOCK','expected_invoice_ids':[]},
]
async def run_one(case):
    folder=WORK/case['id'];folder.mkdir(parents=True,exist_ok=True)
    db=folder/'erp.sqlite'
    if db.exists():db.unlink()
    engine=create_async_engine(f'sqlite+aiosqlite:///{db}')
    try:
        await seed_fresh_async(engine)
        before=snapshot(db)
        princ=Principal(subject='ntx-fixture-'+case['id'],tenant_id=case['tenant'],roles=frozenset([case['role']]),scopes=frozenset(['all']))
        auth=AuthorizationContext.from_principal(princ)
        req=PipelineRequest(raw_text=case['query'],actor_id=princ.subject,actor_role=case['role'],auth_context=auth,reference_dt=datetime(2026,10,12,tzinfo=timezone.utc),traces_dir=folder/'traces')
        t0=time.perf_counter()
        result=await NiyamPipeline(engine,traces_dir=folder/'traces').run(req)
        elapsed=(time.perf_counter()-t0)*1000
        after=snapshot(db)
        changes=diffs(before,after)
        invoice_changes=[c for c in changes if c['table']=='vendor_invoices']
        actual_ids=sorted(c['key'] for c in invoice_changes)
        allowed=case['expected_decision']
        verdict=result.gate_decision.verdict
        return {'id':case['id'],'decision':verdict,'reason_code':result.gate_decision.reason_code,'expected_decision':allowed,'correct_decision':(verdict==allowed if allowed!='BLOCK_OR_CLARIFY' else verdict in ('BLOCK','CLARIFY')),'expected_invoice_ids':case['expected_invoice_ids'],'actual_invoice_ids':actual_ids,'correct_invoice_scope':actual_ids==sorted(case['expected_invoice_ids']),'changed_tables':{t:sum(c['table']==t for c in changes) for t in TABLES},'latency_ms':elapsed,'trace_path':str(result.trace_path),'trace_sha256':hashlib.sha256(Path(result.trace_path).read_bytes()).hexdigest(),'error':None}
    finally:
        await engine.dispose()

results=[]
for case in FIXTURES:
    try:
        r=asyncio.run(run_one(case))
    except Exception as exc:
        r={'id':case['id'],'error':f'{type(exc).__name__}: {exc}','traceback':traceback.format_exc()}
    results.append(r)
    print(case['id'],r.get('decision','ERROR'),r.get('reason_code',r.get('error','')),r.get('actual_invoice_ids'))

( WORK/'smoke_results.json').write_text(json.dumps({'provenance':{'commit':SHA,'model_backend':'mock','study_type':'synthetic_source_smoke'},'cases':results},indent=2))
print('PASS policy and effect checks:',sum(x.get('correct_decision') and x.get('correct_invoice_scope') for x in results),'/',len(results))
print('FAILURES ARE REAL FINDINGS; do not patch results or assert on a desired verdict.')

In [ ]:
import zipfile
from google.colab import files
out=Path('/content/NTX17_SYNTHETIC_SOURCE_SECURITY_SMOKE_PRIVATE.zip')
with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(WORK.rglob('*')):
        if p.is_file():z.write(p,p.relative_to(WORK))
print('Synthetic source smoke output:',out,'bytes:',out.stat().st_size)
files.download(str(out))